In [ ]:
import os
import pickle
import mediapipe as mp
import cv2
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- Initialisation Mediapipe ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3)

DATA_DIR = './data_sign_language'

data = []
labels = []

print("Traitement des images en cours...")

# Parcours des dossiers d'images
for dir_ in os.listdir(DATA_DIR):
    for img_path in os.listdir(os.path.join(DATA_DIR, dir_)):
        data_aux = []
        x_ = []
        y_ = []

        img = cv2.imread(os.path.join(DATA_DIR, dir_, img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = hands.process(img_rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                # Récupérer toutes les coordonnées X et Y pour normalisation
                for i in range(len(hand_landmarks.landmark)):
                    x = hand_landmarks.landmark[i].x
                    y = hand_landmarks.landmark[i].y
                    x_.append(x)
                    y_.append(y)

                # Normalisation : on soustrait le min pour centrer la main (invariance de position)
                for i in range(len(hand_landmarks.landmark)):
                    x = hand_landmarks.landmark[i].x
                    y = hand_landmarks.landmark[i].y
                    data_aux.append(x - min(x_))
                    data_aux.append(y - min(y_))

            # On ajoute seulement si une main est détectée
            # Attention : ce code gère 1 seule main. 
            # Si data_aux a la bonne taille (42 valeurs pour 21 points x 2 coords), on ajoute
            if len(data_aux) == 42: 
                data.append(data_aux)
                labels.append(dir_)

# Conversion en tableaux Numpy
data = np.asarray(data)
labels = np.asarray(labels)

# --- Entraînement ---
# Séparation train/test
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

# Modèle : Random Forest
model = RandomForestClassifier()
model.fit(x_train, y_train)

# Évaluation
y_predict = model.predict(x_test)
score = accuracy_score(y_predict, y_test)

print(f'{score * 100:.2f}% de précision sur les échantillons de test !')

# Sauvegarde du modèle
f = open('model_sign_language.p', 'wb')
pickle.dump({'model': model}, f)
f.close()
print("Modèle sauvegardé sous 'model_sign_language.p'")